<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# The Main Data Loading Pipeline Summarized

The complete chapter code is located in [ch02.ipynb](./ch02.ipynb).

This notebook contains the main takeaway, the data loading pipeline without the intermediate steps.

Packages that are being used in this notebook:

In [1]:
# NBVAL_SKIP
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.5.0+cu124
tiktoken version: 0.8.0


In [2]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

vocab_size = 50257 
output_dim = 256 #emb_dim
context_length = 1024


token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length)

In [3]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

In [4]:
print(input_embeddings.shape) #[batch_size, context_length, output_dim]

torch.Size([8, 4, 256])


In [9]:
x.shape

torch.Size([8, 4])

In [6]:
y

tensor([[  502,   466,   340,    13],
        [  470,   954,    11,   780],
        [ 1639,  1683,  2993,    30],
        [  691, 12226,   318,   284],
        [ 1969,  2157,   502,    11],
        [ 1498,   284,   910,   644],
        [  198,  1544, 45111,  2241],
        [  526,   198,   198,  5779]])

In [7]:
input_embeddings

tensor([[[ 1.9992, -2.9375,  1.7644,  ..., -3.3228,  2.4788, -0.7656],
         [-1.7451,  1.5840,  1.3703,  ..., -0.1882,  2.8001, -0.2332],
         [-0.4897,  2.1347, -1.5434,  ..., -0.1586, -2.7908,  0.6756],
         [-2.5759, -2.0647,  0.2199,  ...,  0.1857,  0.4393, -1.7513]],

        [[ 1.3834, -2.3031,  1.8389,  ..., -3.7937,  1.1451,  0.1786],
         [-0.9108, -0.5867, -0.2226,  ..., -1.6114,  0.1231, -0.3246],
         [-1.1983, -0.5295, -0.1617,  ..., -0.4202,  0.0664, -1.1307],
         [-0.4086,  0.0409, -0.6307,  ...,  1.0397, -1.5140, -0.3271]],

        [[ 3.4734,  0.2591,  0.1162,  ..., -2.6366,  2.5108, -2.4047],
         [ 0.4966,  0.8952,  0.0577,  ..., -2.1398,  0.6341,  1.4597],
         [ 0.6383,  2.2544, -0.0496,  ...,  0.1826, -1.8737,  1.0962],
         [ 0.3966,  0.3011,  1.3216,  ...,  1.2738, -1.4847, -1.3283]],

        ...,

        [[ 2.7943, -0.9396,  0.6418,  ..., -3.5243, -0.1290, -0.8403],
         [ 0.5099, -2.4127, -0.2416,  ..., -1.9879,  2.44